# 65 — W5 SID weight sweep on dev

Sweeps SID stream weight in wRRF over {0.3, 0.5, 0.7, 1.0}, runs each variant
on the dev split, computes paired-bootstrap CIs vs the W4 baseline (config 170
with weight=0.5), picks the winner.

Wallclock: ~5-6 hr on L4 (4 full-pipeline runs back-to-back) / ~2-3 hr Blackwell.


In [ ]:
# 1) Setup — same pattern as notebook 64.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('sid_training', 'recsys2026_sid_training_cache'),
    ('dense', 'recsys2026_dense_cache'),
    ('w5_sweep', 'recsys2026_w5_sweep_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q -U "peft>=0.10" "transformers>=4.40" "accelerate>=0.30" "torchao>=0.17"


In [ ]:
# 2) Generate 4 ephemeral configs by copying 170 + injecting sid_stream_weight overrides.
import datetime
from omegaconf import OmegaConf

%cd /content/recsys2026/music-crs-baselines
WEIGHTS = [0.3, 0.5, 0.7, 1.0]
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
print(f'Sweep run_id: {RUN_ID}')

base_config = OmegaConf.load('config/170-wrrf-sid-v5kto-blindsetA.yaml')
# Override test_dataset_name to point at dev split (TalkPlayData test split = dev set)
base_config.test_dataset_name = 'talkpl-ai/TalkPlayData-Challenge-Dataset'

for w in WEIGHTS:
    cfg = OmegaConf.merge(base_config, OmegaConf.create({'sid_stream_weight': w}))
    tid = f'170-sweep-w{int(w*10):02d}'
    OmegaConf.save(cfg, f'config/{tid}.yaml')
    print(f'  wrote config/{tid}.yaml (sid_stream_weight={w})')
# Preflight: confirm baselines exist before kicking off 5-6 hr of inference.
# Both must come from prior runs (notebook 64 for 170, earlier work for 132).
%cd /content/recsys2026
import os
required_baselines = [
    'music-crs-baselines/exp/inference/dev/170-wrrf-sid-v5kto-blindsetA.json',
    'music-crs-baselines/exp/inference/dev/132-bge-m3-v5kto-prorank-rerank-blindsetA.json',
]
missing = [p for p in required_baselines if not os.path.exists(p)]
assert not missing, (
    f'Missing baseline dev predictions: {missing}. '
    f'Run notebook 64 (W4 dev gate) for 170 and notebook 50 (or wherever 132 was last run on dev) before starting the sweep.'
)
print('OK: both baseline dev predictions present — sweep can proceed')
%cd /content/recsys2026/music-crs-baselines


In [ ]:
# 3) Run each weight variant on dev. ~75 min each on L4. (Skip variants you've already run.)
import os
for w in WEIGHTS:
    tid = f'170-sweep-w{int(w*10):02d}'
    out_path = f'exp/inference/dev/{tid}.json'
    if os.path.exists(out_path):
        print(f'SKIP (cached): {tid}')
        continue
    print(f'RUN: {tid} (weight={w})')
    !python run_inference_blindset.py \
        --tid {tid} \
        --eval_dataset dev \
        --batch_size 32 \
        2>&1 | tee /content/drive/MyDrive/recsys2026_w5_sweep_cache/{RUN_ID}_{tid}.log | tail -30

In [ ]:
# 4) Paired-bootstrap CIs: each weight variant vs config 170 (weight=0.5 baseline).
%cd /content/recsys2026
import json, os
os.makedirs('experiments/cache/w5_sweep', exist_ok=True)
sweep_results = {}
for w in WEIGHTS:
    tid = f'170-sweep-w{int(w*10):02d}'
    out_path = f'experiments/cache/w5_sweep/{RUN_ID}_{tid}_vs_170.json'
    label_a = f'sweep w={w}'
    !python scripts/compare_blind_predictions.py \\
        --pred_a music-crs-baselines/exp/inference/dev/{tid}.json \\
        --pred_b music-crs-baselines/exp/inference/dev/170-wrrf-sid-v5kto-blindsetA.json \\
        --dataset talkpl-ai/TalkPlayData-Challenge-Dataset \\
        --gold_split test \\
        --label_a "{label_a}" \\
        --label_b "baseline w=0.5 (170)" \\
        --n_resamples 1000 \\
        --alpha 0.05 \\
        --output {out_path}
    sweep_results[w] = json.load(open(out_path))

In [ ]:
# 5) Pick winner: highest mean_ndcg_a among the 4 variants. Print table + persist summary.
print(f"{'weight':>6} {'mean_ndcg':>10} {'delta':>10} {'CI lo':>8} {'CI hi':>8}")
print('-' * 50)
best_w, best_ndcg = None, -1
for w, m in sweep_results.items():
    ndcg = m['mean_ndcg_a']
    delta = m['delta_mean']
    lo = m['paired_bootstrap_ci']['lo']
    hi = m['paired_bootstrap_ci']['hi']
    print(f'{w:>6.1f} {ndcg:>10.4f} {delta:>+10.4f} {lo:>+8.4f} {hi:>+8.4f}')
    if ndcg > best_ndcg:
        best_ndcg, best_w = ndcg, w
print('-' * 50)

# Detect "all variants regress vs baseline" — refuse to recommend a winner.
beats_baseline = [w for w, m in sweep_results.items() if m['delta_mean'] > 0]
if not beats_baseline:
    print('\nWARN: ALL 4 variants regress vs baseline 170 (w=0.5).')
    print('Do NOT proceed with notebook 66 — pivot back to W3 retraining instead.')
    print('Setting best_weight=None to fail-fast in notebook 66.')
    best_w = None
    best_ndcg = None
else:
    print(f'\nWINNER: weight={best_w} with mean_ndcg@20={best_ndcg:.4f}')

summary = {
    'run_id': RUN_ID,
    'best_weight': best_w,
    'best_ndcg': best_ndcg,
    'all_results': {str(w): m for w, m in sweep_results.items()},
}
with open('experiments/cache/w5_sweep/sweep_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'wrote experiments/cache/w5_sweep/sweep_summary.json')

In [ ]:
# 6) Sanity check: best sweep variant vs current champion config 132 on dev.
if best_w is None:
    print('Skipping cell 6 — no winning weight (all variants regressed).')
else:
    best_tid = f'170-sweep-w{int(best_w*10):02d}'
    label_best = f'best (w={best_w})'
    !python scripts/compare_blind_predictions.py \\
        --pred_a music-crs-baselines/exp/inference/dev/{best_tid}.json \\
        --pred_b music-crs-baselines/exp/inference/dev/132-bge-m3-v5kto-prorank-rerank-blindsetA.json \\
        --dataset talkpl-ai/TalkPlayData-Challenge-Dataset \\
        --gold_split test \\
        --label_a "{label_best}" \\
        --label_b "champion 132" \\
        --output experiments/cache/w5_sweep/best_vs_132.json
    print()
    print(json.dumps(json.load(open('experiments/cache/w5_sweep/best_vs_132.json')),
                     indent=2))

## After the sweep

The winning weight is in `experiments/cache/w5_sweep/sweep_summary.json` under `best_weight`.

**Decision tree for Blind-A submissions (notebook 66):**

1. **Winner ensemble beats config 132 on dev (Δ > 0 + CI lo > 0)**: submit the winner ensemble + config 171 (pure-SID) to Blind-A in notebook 66.
2. **Winner ensemble ties 132 on dev**: still submit (Blind-A may differ from dev). Pure-SID 171 still informative.
3. **Winner ensemble loses to 132 on dev**: do NOT submit Blind-A. Pivot: retrain W3 with bigger model OR fix doc2query coverage.

Pre-W6 deadline: 2026-06-20 — leaves 2-day buffer for cleanup + memory writes before Blind-B opens 2026-06-23.
